In [1]:
import os
import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, binomtest
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import brier_score_loss, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score,  mean_absolute_error, mean_squared_error, roc_auc_score

In [2]:
def scalar_from_list(x): 
    if isinstance(x, str): 
        return float(eval(x)[0]) 
    if isinstance(x, list): 
        return float(x[0]) 
    return float(x) 


Models Results

In [3]:
task = "BIN"  # "REG" or "BIN"
model_names = [
    "GRU_LogMel",
    "GRU_Wav",
    "TRANS_LogMel",
    "TRANS_Wav",

    "COLA_Alex_LogMel_Full",
    "COLA_Alex_Wav_Full",

    "COLA_Alex_LogMel_Trait",
    "COLA_Alex_LogMel_Inter",
    "COLA_Alex_LogMel_DELTAH1",

    "COLA_Speech"]
dic_path = {name: f"/workspace/app/planilhas/TaCoLa/{task}_{name}.xlsx"
    for name in model_names}

Summaries

In [4]:
reg_results = []
bin_results = []

for name, path in dic_path.items():
    if not os.path.isfile(path):
        print(f"[SKIP] File not found: {path}")
        continue

    d = pd.read_excel(path)
    print(name)

    if task == "REG":
        y_true = d["y_true"].apply(scalar_from_list).astype(float).values
        y_pred = d["y_pred"].apply(scalar_from_list).astype(float).values

        global_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        global_mae = mean_absolute_error(y_true, y_pred)

        reg_results.append({
            "Model": name,
            "RMSE": global_rmse,
            "MAE": global_mae
        })

        print("Global RMSE:", global_rmse)
        print("Global MAE:", global_mae)
        print()

    else:
        y_true = pd.to_numeric(
            d["y_patient_true"], errors="coerce"
        ).values
        y_prob = pd.to_numeric(
            d["p_patient"], errors="coerce"
        ).values

        # Keep folds with valid targets and probabilities.
        valid = np.isfinite(y_true) & np.isfinite(y_prob)
        y_true = y_true[valid].astype(int)
        y_prob = y_prob[valid].astype(float)

        # Recompute predictions using the fixed threshold.
        y_pred = (y_prob >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).ravel()

        specificity = (
            tn / (tn + fp) if (tn + fp) > 0 else np.nan
        )
        recall_pos = (
            tp / (tp + fn) if (tp + fn) > 0 else np.nan
        )
        auroc = (
            roc_auc_score(y_true, y_prob)
            if len(np.unique(y_true)) == 2 else np.nan
        )

        tau_col = (
            "tau_calibrated"
            if "tau_calibrated" in d.columns else "tau"
        )

        bin_results.append({
            "Model": name,
            "N": len(y_true),
            "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
            #"Macro_F1": f1_score(
            #    y_true, y_pred, average="macro", zero_division=0
            #),
            "Recall_Pos": recall_pos,
            "Specificity": specificity,
            "Precision_Pos": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "F1_Pos": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "AUROC": auroc,
            "Mean_Tau": (
                pd.to_numeric(d[tau_col], errors="coerce").mean()
                if tau_col in d.columns else 0.5
            ),
            "Std_Tau": (
                pd.to_numeric(d[tau_col], errors="coerce").std()
                if tau_col in d.columns else 0.0
            )
        })

        print("AUROC:", auroc)
        print(
            pd.DataFrame({
                "y_true": y_true,
                "p_patient": y_prob
            }).groupby("y_true")["p_patient"].describe()
        )
        print()

if task == "REG":
    df_results = pd.DataFrame(reg_results)
    df_results = df_results.sort_values("RMSE", ascending=True)
else:
    df_results = pd.DataFrame(bin_results)
    df_results = df_results.sort_values(
        ["Balanced_Accuracy", "Recall_Pos", "Specificity", "AUROC"],
        ascending=False
    )

print(df_results)

# df_results.to_excel(f"/workspace/app/planilhas/{task}_ALL_MODELS_COMPARISON.xlsx", index=False)

GRU_LogMel
AUROC: 0.6608
        count      mean       std       min       25%       50%       75%  \
y_true                                                                      
0        25.0  0.441560  0.329520  0.021954  0.087344  0.593166  0.733540   
1        25.0  0.653444  0.205012  0.052876  0.649990  0.718785  0.784545   

             max  
y_true            
0       0.891557  
1       0.820307  

GRU_Wav
AUROC: 0.4224
        count      mean       std       min       25%       50%       75%  \
y_true                                                                      
0        25.0  0.529181  0.071422  0.485570  0.496783  0.499977  0.514736   
1        25.0  0.508681  0.128712  0.092743  0.496051  0.499989  0.506044   

             max  
y_true            
0       0.777806  
1       0.856220  

TRANS_LogMel
AUROC: 0.5632
        count      mean       std       min       25%       50%       75%  \
y_true                                                                      


___

Paired Bootstrap

Estimates confidence intervals for performance differences between models using ONE SEED

In [ ]:
all_results = []
for name, path in dic_path.items():
    if not os.path.isfile(path):
        print(f"[SKIP] File not found: {path}")
        continue
    d = pd.read_excel(path).copy()
    d["Experiment"] = name
    all_results.append(d)
df_all_results = pd.concat(all_results, ignore_index=True, sort=False)
#df_all_results.to_excel(f"/workspace/app/planilhas/{task}_ALL_MODEL_RESULTS.xlsx",index=False)
print(df_all_results.shape)
print(df_all_results[["Experiment", "Patient_ID"]].head())

In [ ]:
def paired_predictions(df, ref_name, base_name, model_col="Experiment"):
    cols = ["Patient_ID", "y_patient_true", "y_patient_pred", "p_patient"]
    ref = df[df[model_col] == ref_name][cols].copy()
    base = df[df[model_col] == base_name][cols].copy()

    ref = ref.rename(columns={
        "y_patient_true": "y_true_ref",
        "y_patient_pred": "y_pred_ref",
        "p_patient": "y_prob_ref"
    })
    base = base.rename(columns={
        "y_patient_true": "y_true_base",
        "y_patient_pred": "y_pred_base",
        "p_patient": "y_prob_base"
    })

    paired = ref.merge(base, on="Patient_ID", how="inner", validate="one_to_one")
    numeric_cols = [c for c in paired.columns if c != "Patient_ID"]
    paired[numeric_cols] = paired[numeric_cols].apply(pd.to_numeric, errors="coerce")
    paired = paired.dropna(subset=numeric_cols).copy()
    if not np.array_equal(paired["y_true_ref"].values, paired["y_true_base"].values):
        raise ValueError("The true labels differ between models.")
    return paired

In [ ]:
def classification_metric(y_true, y_pred, y_prob, metric):
    if metric == "Balanced_Accuracy":
        return balanced_accuracy_score(y_true, y_pred)
    #if metric == "Macro_F1":
    #    return f1_score(y_true, y_pred, average="macro", zero_division=0)
    if metric == "Recall_Pos":
        return recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    if metric == "Specificity":
        return recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    if metric == "AUROC":
        return roc_auc_score(y_true, y_prob)
    raise ValueError(f"Unknown metric: {metric}")

In [ ]:
def paired_bootstrap_metric(paired, metric, n_boot=2000, seed=42):
    y_true = paired["y_true_ref"].astype(int).values
    pred_ref = paired["y_pred_ref"].astype(int).values
    pred_base = paired["y_pred_base"].astype(int).values
    prob_ref = paired["y_prob_ref"].astype(float).values
    prob_base = paired["y_prob_base"].astype(float).values

    ref_score = classification_metric(y_true, pred_ref, prob_ref, metric)
    base_score = classification_metric(y_true, pred_base, prob_base, metric)
    rng = np.random.default_rng(seed)
    differences = []

    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        y_boot = y_true[idx]

        if metric == "AUROC" and len(np.unique(y_boot)) < 2:
            continue

        ref_boot = classification_metric(
            y_boot, pred_ref[idx], prob_ref[idx], metric)
        base_boot = classification_metric(
            y_boot, pred_base[idx], prob_base[idx], metric)

        differences.append(ref_boot - base_boot)

    differences = np.asarray(differences)
    lo, hi = np.percentile(differences, [2.5, 97.5])

    return {
        "Ref_Score": ref_score,
        "Base_Score": base_score,
        "Difference": ref_score - base_score,
        "CI_Lower": lo,
        "CI_Upper": hi
    }

In [ ]:
model_ref = "COLA_Alex_LogMel_Full"
baselines = ["GRU_LogMel", "TRANS_LogMel"]
metrics = ["Balanced_Accuracy", "Recall_Pos", "Specificity", "AUROC"]

bootstrap_results = []

for base in baselines:
    paired = paired_predictions(df_all_results, model_ref, base, model_col="Experiment")
    print(f"\n{model_ref} vs {base} | N={len(paired)}")
    for metric in metrics:
        result = paired_bootstrap_metric(paired, metric, n_boot=2000, seed=42)
        bootstrap_results.append({
            "Reference": model_ref,
            "Baseline": base,
            "Metric": metric,
            "N": len(paired),
            **result
        })
        print(
            f"{metric}: {result['Ref_Score']:.3f} vs "
            f"{result['Base_Score']:.3f} | "
            f"Δ={result['Difference']:.3f} "
            f"[95% CI {result['CI_Lower']:.6f}, "
            f"{result['CI_Upper']:.6f}]"
        )
df_bootstrap = pd.DataFrame(bootstrap_results)